# 01 - EDA
This notebook explores dataset structure, class balance, image properties, and pixel statistics.

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

DATA_DIR = Path("../data/chest_xray")
assert DATA_DIR.exists(), "Dataset missing: download and extract first."
sns.set_style("whitegrid")

In [ ]:
splits = ["train", "val", "test"]
classes = ["NORMAL", "PNEUMONIA"]
counts = {}
for split in splits:
    counts[split] = {}
    for cls in classes:
        counts[split][cls] = len(list((DATA_DIR/split/cls).glob("*")))
counts

In [ ]:
import pandas as pd
df = pd.DataFrame(counts).T
df.plot(kind="bar", figsize=(8,5), title="Class Distribution by Split")
plt.ylabel("Image Count")
plt.tight_layout(); plt.savefig("../results/eda_class_distribution.png", dpi=180)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 7))
for r, cls in enumerate(classes):
    img_paths = list((DATA_DIR/"train"/cls).glob("*"))[:3]
    for c, p in enumerate(img_paths):
        img = Image.open(p).convert("L")
        axes[r, c].imshow(img, cmap="gray")
        axes[r, c].set_title(f"{cls}: {p.name[:20]}")
        axes[r, c].axis("off")
plt.tight_layout(); plt.savefig("../results/eda_sample_grid.png", dpi=180)

In [ ]:
sizes = []
pixels = []
for cls in classes:
    for p in list((DATA_DIR/"train"/cls).glob("*"))[:300]:
        arr = np.array(Image.open(p).convert("L"))
        sizes.append(arr.shape)
        pixels.append(arr.flatten())
pixels = np.concatenate(pixels)
heights = [s[0] for s in sizes]
widths = [s[1] for s in sizes]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(heights, bins=20, alpha=0.7, label="Height")
axes[0].hist(widths, bins=20, alpha=0.7, label="Width")
axes[0].legend(); axes[0].set_title("Image Size Distribution")
axes[1].hist(pixels, bins=50, color="gray")
axes[1].set_title("Pixel Intensity Distribution")
plt.tight_layout(); plt.savefig("../results/eda_size_pixel_distribution.png", dpi=180)

## Imbalance Discussion
- The dataset is skewed toward **PNEUMONIA** in training data.
- We address this during training using `class_weight`.
- We evaluate with weighted precision/recall/F1 and AUC-ROC to reduce bias from imbalance.